# Graph Neural Networks for Molecular Property Prediction (Tox21)
## Exploratory Data Analysis & GNN Walkthrough
This notebook demonstrates SMILES graph conversion, Bemis-Murcko scaffold splitting, GNN model training, and gradient atom saliency visualization.

In [1]:
import os
import sys
sys.path.append('..')

import torch
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.featurize import smiles_to_graph
from src.dataset import Tox21GraphDataset, TOX21_TASKS
from src.model import get_model
from src.train import train_gnn
from src.visualize import compute_atom_saliency, render_molecule_svg

### 1. Load Tox21 Dataset & Inspect Bioassay Labels

In [2]:
dataset = Tox21GraphDataset(root_dir='../data')
print(f"Total Molecules Loaded: {len(dataset.df)}")
dataset.df[TOX21_TASKS].describe()

Loaded Tox21 dataframe with 7831 molecules and 12 tasks.
Loading pre-processed molecular graphs from cache...
Loaded 7823 valid cached graphs (72-dim node features).
Total Molecules Loaded: 7831


,NR-AR,NR-AR-LBD,NR-AhR,NR-Aromatase,NR-ER,NR-ER-LBD,NR-PPAR-gamma,SR-ARE,SR-ATAD5,SR-HSE,SR-MMP,SR-p53
count,7265.000000,6758.000000,6549.000000,5821.000000,6193.000000,6955.000000,6450.000000,5832.000000,7072.000000,6467.000000,5810.000000,6774.000000
mean,0.042533,0.035070,0.117270,0.051538,0.128048,0.050324,0.028837,0.161523,0.037330,0.057523,0.158003,0.062445
std,0.201815,0.183969,0.321766,0.221110,0.334170,0.218627,0.167362,0.368044,0.189583,0.232857,0.364776,0.241979
min,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
25%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
50%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
75%,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
max,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000


### 2. Molecular Graph Featurization Test

In [3]:
aspirin_smiles = "CC(=O)OC1=CC=CC=C1C(=O)O"
graph = smiles_to_graph(aspirin_smiles)
print(f"Node Matrix (x) Shape: {graph.x.shape}")
print(f"Edge Index Shape: {graph.edge_index.shape}")
print(f"Edge Attributes Shape: {graph.edge_attr.shape}")

Node Matrix (x) Shape: torch.Size([13, 72])
Edge Index Shape: torch.Size([2, 26])
Edge Attributes Shape: torch.Size([26, 6])


### 3. Bemis-Murcko Scaffold Splitting

In [4]:
train_g, val_g, test_g = dataset.get_scaffold_splits()
print(f"Train Molecules: {len(train_g)}, Val: {len(val_g)}, Test: {len(test_g)}")

[09:47:43] WARNING: not removing hydrogen atom without neighbors


Scaffold Split Summary -> Train: 6258, Val: 782, Test: 783
Train Molecules: 6258, Val: 782, Test: 783


### 4. Train GCN Model

In [5]:
results = train_gnn(arch='gcn', epochs=10, batch_size=64, save_dir='../models')
print(f"Test ROC-AUC: {results['test_auc']:.4f}")

Download complete.
Loaded Tox21 dataframe with 7831 molecules and 12 tasks.
Converting SMILES to molecular graphs...


 28%|██▊       | 2199/7831 [00:01<00:03, 1482.19it/s][09:47:51] Explicit valence for atom # 3 Al, 6, is greater than permitted
[09:47:51] Explicit valence for atom # 4 Al, 6, is greater than permitted
100%|██████████| 7831/7831 [00:07<00:00, 1069.06it/s]


Successfully processed 7823 / 7831 molecules.


[09:47:58] WARNING: not removing hydrogen atom without neighbors


Scaffold Split Summary -> Train: 6258, Val: 782, Test: 783

 Starting Training for GCN GNN
 Device          : cpu
 Trainable Params: 52,172
 Architecture    : GCN (Hidden Dim: 128, In: 72, Edge: 6)
 Hyperparameters : Epochs: 10 | Batch Size: 64 | Initial LR: 0.001
 Scaffold Splits : Train: 6258 graphs (98 batches) | Val: 782 graphs (13 batches) | Test: 783 graphs (13 batches)
 Pos Imbalance   : weights range [5.66 - 10.00] across 12 endpoints
-------------------------------------------------------


...

Epoch 07/10 | Train Loss: 0.6904 | Val ROC-AUC: 0.7265 | LR: 1.0e-03 | Time: 3.5s


Epoch 08/10 | Train Loss: 0.6774 | Val ROC-AUC: 0.7411 | LR: 1.0e-03 | Time: 3.5s [Saved Best Model: Val AUC = 0.7411]


Epoch 09/10 | Train Loss: 0.6706 | Val ROC-AUC: 0.7516 | LR: 1.0e-03 | Time: 3.4s [Saved Best Model: Val AUC = 0.7516]


Epoch 10/10 | Train Loss: 0.6627 | Val ROC-AUC: 0.7384 | LR: 1.0e-03 | Time: 3.5s

Evaluating Best Checkpoint on Test Set (783 molecules)...


Saved training curves figure to results\figures\training_curves_gcn.png
-------------------------------------------------------
Final Test Mean ROC-AUC for GCN: 0.7293
-------------------------------------------------------

Test ROC-AUC: 0.7293
